# V15.22 Automated Pipeline Runner

**Purpose:** Run the full V15.22 Critical Controls experiment automatically.

**Setup (one-time):**
1. Sync repo to Google Drive: `./scripts/sync_to_drive.sh`
2. Open this notebook from Google Drive in Colab

**Prerequisites:**
- Google Drive with `paladin_claude` folder synced
- OpenAI API key in Colab Secrets (name: `OPENAI_API_KEY`) - optional
- GPU runtime enabled (T4 or A100)

**Estimated Runtime:** ~10-13 GPU-hours

---

## 1. Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {vram:.1f} GB")
    if vram < 14:
        print("Note: 8-bit quantization will be used for T4")

In [ ]:
# Install dependencies
!pip install transformers accelerate bitsandbytes scipy openai pyyaml tqdm -q
print("Dependencies installed.")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# Find paladin_claude folder in Drive
DRIVE_BASE = Path('/content/drive/MyDrive')
REPO_DIR = DRIVE_BASE / 'paladin_claude'

if not REPO_DIR.exists():
    print(f"ERROR: {REPO_DIR} not found!")
    print("\nTo set up:")
    print("  1. On your local machine, run: ./scripts/sync_to_drive.sh")
    print("  2. Wait for Google Drive to sync")
    print("  3. Re-run this cell")
else:
    os.chdir(REPO_DIR)
    print(f"Working directory: {os.getcwd()}")
    print(f"\nContents:")
    !ls -la

In [ ]:
# Create output directories
OUTPUT_DIR = REPO_DIR / 'v1522_results'
CHECKPOINT_DIR = REPO_DIR / 'v1522_checkpoints'

OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR.mkdir(exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

## 3. Configure API Keys (Optional)

In [ ]:
# Load OpenAI API key for GPT-4 coherence judging (optional)
import os

USE_GPT4 = False

try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
    if api_key:
        os.environ['OPENAI_API_KEY'] = api_key
        print("OpenAI API key loaded from Colab Secrets")
        USE_GPT4 = True
    else:
        print("No OpenAI API key found - will use heuristic coherence metrics")
except Exception as e:
    print(f"OpenAI API key not available - will use heuristic coherence metrics")
    print(f"(To enable GPT-4 judging: Colab Secrets > Add OPENAI_API_KEY)")

## 4. Run Configuration

In [ ]:
import yaml

# Configuration options
CONFIG = {
    # Model
    "model_name": "google/gemma-2-9b-it",
    "target_layer": 21,
    "alpha": -3.0,
    "use_8bit": True,
    
    # Control 1: Direction Specificity
    "n_random": 10,
    "n_ortho": 5,
    "rotation_angles": [5, 10, 15, 20, 45],
    "n_prompts_control1": 20,
    
    # Control 2: Coherence
    "use_gpt4_judge": USE_GPT4,
    "openai_model": "gpt-4-turbo",
    "n_prompts_control2": 10,
    
    # Control 3: Statistical Power
    "n_prompts_control3": 50,
    "n_benign": 20,
    
    # Thresholds
    "threshold_random_pass": 0.20,
    "threshold_random_fail": 0.50,
    "threshold_coherence_high": 4.0,
    "threshold_coherence_low": 2.5,
    "threshold_flip_rate": 0.50,
    "threshold_coherent_flip": 0.30,
    "threshold_benign_degradation": 0.20,
    
    # Execution
    "seed": 42,
    "output_dir": str(OUTPUT_DIR),
    "checkpoint_dir": str(CHECKPOINT_DIR),
}

# Save config
config_path = REPO_DIR / 'v1522' / 'run_config.yaml'
with open(config_path, 'w') as f:
    yaml.dump(CONFIG, f, default_flow_style=False)

print("Configuration:")
print(f"  Model: {CONFIG['model_name']}")
print(f"  Target layer: {CONFIG['target_layer']}")
print(f"  GPT-4 judge: {CONFIG['use_gpt4_judge']}")
print(f"  Output: {CONFIG['output_dir']}")

## 5. Run Full Pipeline

This runs all 3 controls with automatic checkpointing.
If disconnected, re-run from **Section 5b** to resume.

In [ ]:
# Run the full pipeline
!cd {REPO_DIR} && python v1522/v1522_pipeline.py --config v1522/run_config.yaml

## 5b. Resume from Checkpoint (if disconnected)

In [ ]:
# Resume from checkpoint (run this if the previous cell was interrupted)
!cd {REPO_DIR} && python v1522/v1522_pipeline.py --config v1522/run_config.yaml --resume

## 5c. Run Individual Controls (optional)

In [ ]:
# Uncomment to run individual controls:

# Control 1 only (Direction Specificity) - ~3 hours
# !cd {REPO_DIR} && python v1522/v1522_pipeline.py --config v1522/run_config.yaml --control 1

# Control 2 only (Coherence) - ~2 hours
# !cd {REPO_DIR} && python v1522/v1522_pipeline.py --config v1522/run_config.yaml --control 2

# Control 3 only (Statistical Power n=50) - ~3 hours
# !cd {REPO_DIR} && python v1522/v1522_pipeline.py --config v1522/run_config.yaml --control 3

## 6. View Results

In [ ]:
import json

# Load and display decision summary
summary_path = OUTPUT_DIR / "v1522_decision_summary.json"

if summary_path.exists():
    with open(summary_path) as f:
        summary = json.load(f)
    
    print("=" * 60)
    print("V15.22 CRITICAL CONTROLS - RESULTS")
    print("=" * 60)
    print(f"\nTimestamp: {summary['timestamp']}")
    print(f"Model: {summary['model']}")
    print(f"Target Layer: {summary['target_layer']}")
    print()
    print("Gate Results:")
    print("-" * 40)
    for gate_name, gate_data in summary['gates'].items():
        print(f"  {gate_name}: {gate_data['verdict']}")
        print(f"    {gate_data['interpretation']}")
    print()
    print("=" * 60)
    print(f"FINAL VERDICT: {summary['final_verdict']}")
    print(f"Action: {summary['action']}")
    print("=" * 60)
else:
    print("No results found yet. Run the pipeline first (Section 5).")

In [ ]:
# Display formatted markdown summary
md_path = OUTPUT_DIR / "v1522_decision_summary.md"

if md_path.exists():
    from IPython.display import Markdown, display
    with open(md_path) as f:
        display(Markdown(f.read()))
else:
    print("No markdown summary found.")

In [ ]:
# List all output files
print("Output files:")
print("-" * 40)
if OUTPUT_DIR.exists():
    for f in sorted(OUTPUT_DIR.glob("*")):
        size = f.stat().st_size / 1024
        print(f"  {f.name} ({size:.1f} KB)")
else:
    print("  (no output files yet)")

## 7. Key Metrics Deep Dive

In [ ]:
# Load Control 1 results for detailed analysis
control1_path = OUTPUT_DIR / "v1522_control1.json"

if control1_path.exists():
    with open(control1_path) as f:
        c1 = json.load(f)
    
    print("CONTROL 1: Direction Specificity")
    print("=" * 40)
    print(f"Extracted direction mean effect: {c1['summary']['extracted_mean']:.2f}")
    print(f"Random directions mean effect:   {c1['summary']['random_mean']:.2f}")
    print(f"Orthogonal directions mean:      {c1['summary']['ortho_mean']:.2f}")
    print(f"\nRandom/Extracted ratio: {c1['summary']['random_ratio']:.1%}")
    print(f"Verdict: {c1.get('gate_verdict', 'N/A')}")
else:
    print("Control 1 results not found.")

In [ ]:
# Load Control 3 results
control3_path = OUTPUT_DIR / "v1522_control3.json"

if control3_path.exists():
    with open(control3_path) as f:
        c3 = json.load(f)
    
    print("CONTROL 3: Statistical Power (n=50)")
    print("=" * 40)
    print(f"Flip rate: {c3['summary']['flip_rate']:.1%}")
    print(f"Coherent flip rate: {c3['summary']['coherent_flip_rate']:.1%}")
    print(f"95% CI: [{c3['summary']['ci_95'][0]:.1%}, {c3['summary']['ci_95'][1]:.1%}]")
    print(f"Benign degradation: {c3['summary']['degradation_rate']:.1%}")
    print(f"\nVerdict: {c3.get('gate_verdict', 'N/A')}")
else:
    print("Control 3 results not found.")

---

## Next Steps

### PUBLISH (All GREEN)
1. Review outputs for anomalies
2. Run Tier 1.5 probes (entropy, transfer function)
3. Draft security paper
4. Consider responsible disclosure to Google

### INVESTIGATE (Any YELLOW)
1. Run Tier 1.5 mechanism probes
2. Analyze which tests showed weakness
3. Consider additional controls

### STOP (Any RED)
1. Review failed gate
2. Check logs: `v1522_results/v1522_pipeline.log`
3. Revise hypothesis or methodology

---

*Results are automatically saved to Google Drive and will sync back to your local machine.*